# ⚙️ MLOps & Experiment Tracking

> 💡 **Why this matters:** In 2026, a model that only works in your notebook is not a data science result — it's a prototype. MLOps is the set of practices that make models reliable, reproducible, and maintainable in the real world.


## 1. What is MLOps?

**MLOps** (Machine Learning Operations) is the practice of combining ML development with software engineering and DevOps principles to deploy and maintain models in production reliably.

### The problem MLOps solves

Imagine this scenario — it happens constantly in industry:

```
Month 1: Data scientist builds a model. Accuracy: 94%. 🎉
Month 2: Model deployed. Works great.
Month 4: Accuracy silently drops to 71%. Nobody notices.
Month 6: Business makes bad decisions based on stale model.
Month 7: Data scientist can't reproduce the original model.
         Which dataset version? Which hyperparameters? 😱
```

MLOps prevents this by answering four questions at all times:

| Question | MLOps tool |
|---|---|
| **What experiment produced this model?** | Experiment tracking (MLflow, W&B) |
| **What data was used?** | Data versioning (DVC) |
| **Is the model still performing well?** | Monitoring & alerting |
| **Can I reproduce this result?** | Reproducibility practices |


### The ML Lifecycle

```
┌──────────────────────────────────────────────────────────────────────┐
│                         ML LIFECYCLE                                 │
│                                                                      │
│  ┌──────────┐    ┌──────────┐    ┌──────────┐    ┌──────────┐       │
│  │  Data    │───▶│  Train   │───▶│ Evaluate │───▶│  Deploy  │       │
│  │ Prep &   │    │ & Track  │    │ & Select │    │ & Serve  │       │
│  │Versioning│    │Experiment│    │  Model   │    │  Model   │       │
│  └──────────┘    └──────────┘    └──────────┘    └──────────┘       │
│       │               │                │               │            │
│       ▼               ▼                ▼               ▼            │
│    DVC/Git        MLflow/W&B       Model Registry   FastAPI/        │
│                                    (MLflow)         Docker          │
│                                                          │          │
│  ◀─────────────────── Monitor & Retrain ─────────────────           │
│                    (data drift, performance drop)                    │
└──────────────────────────────────────────────────────────────────────┘
```

Each step feeds back. When monitoring detects a problem, you retrain — and the cycle repeats.


## 2. Experiment Tracking with MLflow

**The problem without tracking:**
```python
# Monday
model_v1 = RandomForestClassifier(n_estimators=100)   # accuracy 0.82

# Tuesday (you forgot what you tried Monday)
model_v2 = RandomForestClassifier(n_estimators=200)   # accuracy 0.85?

# Wednesday (what was best again??)
model_v3 = GradientBoostingClassifier(...)            # ???
```

**MLflow solves this** by logging every experiment automatically:
- Parameters (hyperparameters)
- Metrics (accuracy, F1, RMSE)
- Artifacts (model files, plots)
- Environment (package versions)

### Installation
```bash
pip install mlflow scikit-learn
```


In [2]:
# Install if needed
import subprocess, sys
for pkg in ["mlflow", "scikit-learn"]:
    try:
        __import__(pkg.replace("-","_"))
    except ImportError:
        subprocess.run([sys.executable, "-m", "pip", "install", pkg, "-q"])

import mlflow
import mlflow.sklearn
import numpy as np
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score
from sklearn.preprocessing import StandardScaler

# ── Load dataset and train a simple ML model --- 
data = load_breast_cancer()
X, y = data.data, data.target
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_test_s  = scaler.transform(X_test)

print(f"Dataset: {data.feature_names[0]} ... | {len(X_train)} train, {len(X_test)} test samples")
print(f"Task: Binary classification (malignant vs benign)")

Dataset: mean radius ... | 455 train, 114 test samples
Task: Binary classification (malignant vs benign)


In [7]:
# ── Run multiple experiments and track them with MLflow ── 

mlflow.set_tracking_uri("sqlite:///mlflow.db")  # local SQLite store
mlflow.set_experiment("breast_cancer_classification")

experiments = [
    {
        "name": "logistic_regression",
        "model": LogisticRegression(C=1.0, max_iter=1000),
        "params": {"C": 1.0, "solver": "lbfgs"},
        "uses_scaling": True,
    },
    {
        "name": "random_forest_100",
        "model": RandomForestClassifier(n_estimators=100, max_depth=5, random_state=42),
        "params": {"n_estimators": 100, "max_depth": 5},
        "uses_scaling": False,
    },
    {
        "name": "random_forest_200",
        "model": RandomForestClassifier(n_estimators=200, max_depth=8, random_state=42),
        "params": {"n_estimators": 200, "max_depth": 8},
        "uses_scaling": False,
    },
    {
        "name": "gradient_boosting",
        "model": GradientBoostingClassifier(n_estimators=100, learning_rate=0.1, random_state=42),
        "params": {"n_estimators": 100, "learning_rate": 0.1},
        "uses_scaling": False,
    },
]

results = []
for exp in experiments:
    Xtr = X_train_s if exp["uses_scaling"] else X_train
    Xte = X_test_s  if exp["uses_scaling"] else X_test

    with mlflow.start_run(run_name=exp["name"]):
        # Train
        exp["model"].fit(Xtr, y_train)
        preds = exp["model"].predict(Xte)
        proba = exp["model"].predict_proba(Xte)[:, 1]

        # Calculate metrics
        acc  = accuracy_score(y_test, preds)
        f1   = f1_score(y_test, preds)
        auc  = roc_auc_score(y_test, proba)

        # 📊 LOG TO MLFLOW
        mlflow.log_params(exp["params"])
        mlflow.log_metrics({"accuracy": acc, "f1_score": f1, "roc_auc": auc})
        mlflow.set_tag("model_type", exp["name"])
        mlflow.sklearn.log_model(exp["model"], "model")

        results.append({"name": exp["name"], "accuracy": acc, "f1": f1, "auc": auc})
        print(f"✅ {exp['name']:<25} | acc={acc:.3f} | f1={f1:.3f} | auc={auc:.3f}")

print(f"\n All runs logged to MLflow (sqlite:///mlflow.db)")
print("   Run: mlflow ui  →  open http://localhost:5000 to see the dashboard")

2026/04/17 16:33:57 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/04/17 16:33:57 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html
2026/04/17 16:34:01 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/04/17 16:34:01 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_p

✅ logistic_regression       | acc=0.974 | f1=0.979 | auc=0.997
✅ random_forest_100         | acc=0.965 | f1=0.972 | auc=0.996


2026/04/17 16:34:05 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/04/17 16:34:05 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


✅ random_forest_200         | acc=0.965 | f1=0.972 | auc=0.996


2026/04/17 16:34:09 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/04/17 16:34:09 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


✅ gradient_boosting         | acc=0.956 | f1=0.965 | auc=0.995

 All runs logged to MLflow (sqlite:///mlflow.db)
   Run: mlflow ui  →  open http://localhost:5000 to see the dashboard


In [3]:
# ── Compare experiments programmatically ─────────────────────────────────
import pandas as pd

df = pd.DataFrame(results).sort_values("auc", ascending=False)
df[["name","accuracy","f1","auc"]] = df[["name","accuracy","f1","auc"]].round(4)

print("Experiment comparison (sorted by AUC):")
print(df.to_string(index=False))

best = df.iloc[0]
print(f"\n🏆 Best model: {best['name']}  (AUC: {best['auc']})")
print("\n💡 In MLflow UI you can:")
print("   - Compare runs visually on a chart")
print("   - Filter by any metric")
print("   - Download the best model artifact")
print("   - Tag a run as 'champion' to register it")


Experiment comparison (sorted by AUC):
               name  accuracy     f1    auc
logistic_regression    0.9737 0.9790 0.9974
  random_forest_100    0.9649 0.9722 0.9964
  random_forest_200    0.9649 0.9722 0.9964
  gradient_boosting    0.9561 0.9650 0.9951

🏆 Best model: logistic_regression  (AUC: 0.9974)

💡 In MLflow UI you can:
   - Compare runs visually on a chart
   - Filter by any metric
   - Download the best model artifact
   - Tag a run as 'champion' to register it



## 3. Reproducibility Best Practices

A reproducible experiment can be rerun by anyone, on any machine, and produce the same result.

### Reproducibility checklist

```
✅ Set random seeds everywhere (numpy, sklearn, torch)
✅ Pin dependency versions (requirements.txt / pyproject.toml)
✅ Version your data (DVC)
✅ Log hyperparameters (MLflow)
✅ Use config files, not hardcoded values
✅ Containerise your environment (Docker)
```


In [5]:
# ── Reproducibility in practice ──────────────────────────────────────────
import numpy as np
from sklearn.ensemble import RandomForestClassifier
from sklearn.datasets import make_classification

# ❌ BAD: no seed — results change every run
X, y = make_classification(n_samples=200, n_features=10, random_state=42)  # data is fixed here for demo
bad_model = RandomForestClassifier()  # no random_state!
bad_model.fit(X, y)
print("Without seed:")
print(f"  Prediction[0]: {bad_model.predict(X[:1])[0]}  (changes each run)")

# ✅ GOOD: seeds fixed everywhere
import random
SEED = 42
random.seed(SEED)
np.random.seed(SEED)

good_model = RandomForestClassifier(n_estimators=100, random_state=SEED)
good_model.fit(X, y)
print(f"\nWith seed={SEED}: Prediction[0] = {good_model.predict(X[:1])[0]}  (always the same)")

# ── Config-driven experiments (no hardcoded values) ───────────────────────
print("\n── Config file approach ──")
config = {
    "seed": 42,
    "test_size": 0.2,
    "model": {
        "name": "RandomForestClassifier",
        "params": {"n_estimators": 100, "max_depth": 5, "random_state": 42}
    },
    "data": {
        "path": "data/train_v3.0.csv",
        "dvc_hash": "a3f8d2c1..."
    }
}

import json
print("config.json:")
print(json.dumps(config, indent=2))
print("\n💡 Store this config file in Git alongside your code.")
print("   Anyone can reproduce your exact experiment by running:")
print("   python train.py --config config.json")


Without seed:
  Prediction[0]: 1  (changes each run)

With seed=42: Prediction[0] = 1  (always the same)

── Config file approach ──
config.json:
{
  "seed": 42,
  "test_size": 0.2,
  "model": {
    "name": "RandomForestClassifier",
    "params": {
      "n_estimators": 100,
      "max_depth": 5,
      "random_state": 42
    }
  },
  "data": {
    "path": "data/train_v3.0.csv",
    "dvc_hash": "a3f8d2c1..."
  }
}

💡 Store this config file in Git alongside your code.
   Anyone can reproduce your exact experiment by running:
   python train.py --config config.json


## Summary

| Concept | Tool | What it solves |
|---|---|---|
| **Experiment tracking** | MLflow, W&B | "Which run produced that 94% accuracy?" |
| **Data versioning** | DVC | "Which dataset version was used?" |
| **Reproducibility** | Seeds + config files + Docker | "Can I re-run this from scratch?" |
| **Model registry** | MLflow Registry | "Which model is in production right now?" |
| **Monitoring** | Evidently AI, Arize | "Is the model still performing well?" |
